# 03 — Infrastructure & Physical Shocks

Эта ветка собирает дополнительные источники для физического графа нефтяного рынка.

Источники:

1. **BOEM/BSEE ArcGIS** — offshore platforms, offshore pipelines, planning areas.
2. **BSEE Production by Platform** — месячная offshore production по платформам.
3. **NOAA/NHC HURDAT2** — 6-часовые траектории Atlantic tropical cyclones.
4. **Global Energy Monitor GOIT** — transmission oil pipelines и ownership/status.
5. **OpenStreetMap / Overpass** — gap-filling для refineries и oil storage/terminals.

Все raw-файлы сохраняются в `data/raw/infrastructure/`.


## 1. Environment


In [1]:
import sys
from pathlib import Path

import pandas as pd

print(sys.version)
print(sys.executable)

assert sys.version_info[:2] == (3, 12)
assert ".venv" in sys.executable


3.12.16 (main, Oct 26 2026, 13:59:40) [Clang 23.1.3 ]
/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/.venv/bin/python


## 2. Project paths


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError("Project root not found.")


ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw" / "infrastructure"
PROCESSED_DIR = ROOT / "data" / "processed" / "infrastructure"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)


/Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph


In [3]:
import truststore

truststore.inject_into_ssl()

# 3. BOEM ArcGIS — current spatial reference

BOEM ArcGIS остаётся полезным как **текущий spatial reference**:

- platform points;
- offshore pipeline geometries;
- planning areas.

Важно: BOEM current layer **не используем как единственный historical platform master**.
Исторические install/removal dates и полный набор structures ниже берём из BSEE bulk files.


In [4]:
from src.data.boem_client import BoemClient

boem = BoemClient()

platforms_geojson = boem.fetch_platforms(active_only=False)
pipelines_geojson = boem.fetch_pipelines()
planning_areas_geojson = boem.fetch_planning_areas()

boem.save_geojson(
    platforms_geojson,
    RAW_DIR / "boem_platforms.geojson",
)
boem.save_geojson(
    pipelines_geojson,
    RAW_DIR / "boem_pipelines.geojson",
)
boem.save_geojson(
    planning_areas_geojson,
    RAW_DIR / "boem_planning_areas.geojson",
)

print("BOEM current platforms:", len(platforms_geojson["features"]))
print("BOEM pipeline geometries:", len(pipelines_geojson["features"]))
print("BOEM planning areas:", len(planning_areas_geojson["features"]))


BOEM current platforms: 1315
BOEM pipeline geometries: 17411
BOEM planning areas: 24


In [5]:
boem_platforms_current = boem.points_to_frame(
    platforms_geojson
)

for column in ("INSTALL_DATE", "REMOVAL_DATE"):
    if column in boem_platforms_current.columns:
        boem_platforms_current[column] = pd.to_datetime(
            boem_platforms_current[column],
            unit="ms",
            errors="coerce",
        )

boem_platforms_current.to_parquet(
    PROCESSED_DIR / "boem_platforms_current.parquet",
    index=False,
)

boem_platforms_current.head()


,OBJECTID,COMPLEX_ID_NUM,INSTALL_DATE,REMOVAL_DATE,longitude,latitude
0,17497399,20005,2000-01-01,NaT,-90.125616,29.041021
1,17497403,20008,1958-01-01,NaT,-90.120977,29.043401
2,17497407,20011,1959-01-01,NaT,-90.116217,29.056402
3,17497412,20015,1963-01-01,NaT,-89.819586,28.969793
4,17497416,20020,1968-01-01,NaT,-89.967169,28.993841


# 4. BSEE — historical offshore infrastructure and production

Это основной historical offshore layer.

Собираем:

1. **Platform Structures** — install/removal dates и structure metadata;
2. **Platform Locations** — coordinates;
3. **Production by Platform** — monthly physical production;
4. **Pipeline Masters** — product/current status/attributes;
5. **Pipeline Histories** — dated service events.

Официальный BSEE bulk layer нужен, потому что current BOEM map теряет уже удалённые
structures и сам по себе недостаточен для backtest 2015–2026.


In [6]:
from src.data.bsee_client import BseeClient

bsee = BseeClient()


## 4.1 Historical platform structures + coordinates


In [7]:
platform_structures_zip = bsee.download_platform_structures(
    RAW_DIR / "bsee_platform_structures.zip"
)
platform_locations_zip = bsee.download_platform_locations(
    RAW_DIR / "bsee_platform_locations.zip"
)

platform_structure_files = bsee.extract_archive(
    platform_structures_zip,
    RAW_DIR / "bsee_platform_structures",
)
platform_location_files = bsee.extract_archive(
    platform_locations_zip,
    RAW_DIR / "bsee_platform_locations",
)

platform_structures = bsee.load_platform_structures(
    platform_structure_files
)
platform_locations = bsee.load_platform_locations(
    platform_location_files
)
platform_history = bsee.build_platform_history(
    platform_structures,
    platform_locations,
)

print("structures:", platform_structures.shape)
print("locations:", platform_locations.shape)
print("historical platform table:", platform_history.shape)
print(
    "coordinate coverage:",
    round(platform_history["has_coordinates"].mean(), 4),
)
print(
    "removed structures:",
    platform_history["removal_date"].notna().sum(),
)

platform_history.head()


structures: (7092, 23)
locations: (7306, 14)
historical platform table: (7092, 33)
coordinate coverage: 0.9989
removed structures: 5791


,area_code,block_number,complex_id_num,deck_count,east_west_departure_code,install_date,last_revision_date,major_structure_flag,north_south_departure_code,removal_date,...,district_code,longitude,latitude,x_location,y_location,structure_id,active_from,active_to,is_current,has_coordinates
0,HI,38,2,1,E,1998-01-05,2005-04-07,N,S,2010-06-09,...,4,-93.812831,29.474909,3649872.81,633454.0,2-1,1998-01-05,2010-06-09,False,True
1,HI,176,3,1,W,1997-02-28,2007-09-12,Y,S,2013-08-08,...,4,-94.371776,29.175148,3476500.81,517089.0,3-1,1997-02-28,2013-08-08,False,True
2,WC,263,4,2,E,1997-08-26,2017-02-07,Y,N,2003-06-29,...,5,-93.014599,28.953916,1462252.952,108406.328,4-1,1997-08-26,2003-06-29,False,True
3,WD,40,5,1,W,1997-07-22,2008-08-01,N,S,2007-11-04,...,2,-89.79501,29.068068,2491473.0,149271.3,5-1,1997-07-22,2007-11-04,False,True
4,ST,51,6,2,W,1997-06-19,2023-10-06,N,N,NaT,...,2,-90.504325,28.898662,2265308.02,85327.88,6-1,1997-06-19,NaT,True,True


In [8]:
platform_history.to_parquet(
    PROCESSED_DIR / "bsee_platform_history.parquet",
    index=False,
)


## 4.2 Production by platform — model period only

Full BSEE archive сохраняется как raw source, но в processed не тащим 1940-е–2014.

Делаем два слоя:

- `structure-month` — максимально детальный;
- `complex-month` — удобный вес для hurricane exposure.

**Leakage note:** BSEE production — retrospective monthly data. До отдельной
availability-модели не используем contemporaneous `bopd` как будто он был известен
в реальном времени. Он подходит для historical importance/exposure analysis и позже
должен быть лагирован.


In [9]:
production_zip = bsee.download_production_by_platform(
    RAW_DIR / "bsee_production_by_platform.zip"
)
production_files = bsee.extract_archive(
    production_zip,
    RAW_DIR / "bsee_production_by_platform",
)

production_full = bsee.load_production_by_platform(
    production_files
)
production_2015plus = bsee.prepare_production_since(
    production_full,
    start_date="2015-01-01",
)
production_complex_month = (
    bsee.aggregate_production_by_complex(
        production_2015plus
    )
)

print("full raw rows:", len(production_full))
print("2015+ structure-month rows:", len(production_2015plus))
print("2015+ complex-month rows:", len(production_complex_month))
print(
    "2015+ complexes:",
    production_2015plus["complex_id_num"].nunique(),
)


full raw rows: 1696663
2015+ structure-month rows: 159917
2015+ complex-month rows: 150465
2015+ complexes: 1648


In [10]:
production_2015plus.to_parquet(
    PROCESSED_DIR
    / "bsee_production_by_structure_2015plus.parquet",
    index=False,
)
production_complex_month.to_parquet(
    PROCESSED_DIR
    / "bsee_production_by_complex_month_2015plus.parquet",
    index=False,
)


## 4.3 Historical platform coverage check

Проверяем, какую долю complexes с production 2015+ теперь можно связать с historical
platform master. Это ключевой sanity check перед hurricane exposure.


In [11]:
production_complex_ids = set(
    production_2015plus["complex_id_num"]
    .dropna()
    .astype(int)
)
history_complex_ids = set(
    platform_history["complex_id_num"]
    .dropna()
    .astype(int)
)

matched_complexes = (
    production_complex_ids & history_complex_ids
)
historical_platform_coverage = (
    len(matched_complexes)
    / len(production_complex_ids)
    if production_complex_ids
    else float("nan")
)

print("producing complexes 2015+:", len(production_complex_ids))
print("matched to historical master:", len(matched_complexes))
print("historical coverage:", round(historical_platform_coverage, 4))

assert historical_platform_coverage >= 0.90, (
    "Historical platform coverage is unexpectedly low. "
    "Inspect BSEE Platform Structures/Locations parsing."
)


producing complexes 2015+: 1648
matched to historical master: 1624
historical coverage: 0.9854


## 4.4 Offshore pipeline attributes + history

BOEM даёт geometry, а BSEE Pipeline Masters/Histories добавляют:

- `prod_code` — oil/gas/NGL/etc.;
- `status_code` — current BSEE status;
- construction/abandonment dates;
- dated history events.

**Важно:** BSEE bulk files `pplmastdelimit.txt` и `2100delimit.txt`
читаются как **headerless files**. Pipeline Masters хранит даты как `YYYYMMDD`,
а Pipeline Histories — как `MM/DD/YYYY`, поэтому parser использует разные
форматы дат.

Часть реально встречающихся history codes отсутствует в опубликованном
BSEE enumeration. Такие коды не выбрасываем и не придумываем им смысл:
они сохраняются в `history_code`, а `history_code_documented=False`.

Current status не переносим назад во времени. Для historical graph сохраняем
отдельный event timeline.


In [12]:
pipeline_masters_zip = bsee.download_pipeline_masters(
    RAW_DIR / "bsee_pipeline_masters.zip"
)
pipeline_histories_zip = bsee.download_pipeline_histories(
    RAW_DIR / "bsee_pipeline_histories.zip"
)

pipeline_master_files = bsee.extract_archive(
    pipeline_masters_zip,
    RAW_DIR / "bsee_pipeline_masters",
)
pipeline_history_files = bsee.extract_archive(
    pipeline_histories_zip,
    RAW_DIR / "bsee_pipeline_histories",
)

# BSEE pipeline bulk files are headerless.
# The current BseeClient assigns the official BSEE field names by position.
pipeline_masters = bsee.load_pipeline_masters(
    pipeline_master_files
)
pipeline_histories = bsee.load_pipeline_histories(
    pipeline_history_files
)

required_master_columns = {
    "segment_num",
    "status_code",
    "prod_code",
    "is_crude_oil_related",
}
required_history_columns = {
    "segment_num",
    "history_date",
    "history_code",
}

missing_master_columns = (
    required_master_columns
    - set(pipeline_masters.columns)
)
missing_history_columns = (
    required_history_columns
    - set(pipeline_histories.columns)
)

assert not missing_master_columns, (
    "Pipeline Masters schema problem. "
    f"Missing: {sorted(missing_master_columns)}. "
    f"Columns: {pipeline_masters.columns.tolist()}"
)
assert not missing_history_columns, (
    "Pipeline Histories schema problem. "
    f"Missing: {sorted(missing_history_columns)}. "
    f"Columns: {pipeline_histories.columns.tolist()}"
)

pipeline_masters["segment_num"] = pd.to_numeric(
    pipeline_masters["segment_num"],
    errors="coerce",
).astype("Int64")

pipeline_histories["segment_num"] = pd.to_numeric(
    pipeline_histories["segment_num"],
    errors="coerce",
).astype("Int64")

pipeline_masters = pipeline_masters.dropna(
    subset=["segment_num"]
).copy()
pipeline_histories = pipeline_histories.dropna(
    subset=["segment_num"]
).copy()

print("pipeline masters:", pipeline_masters.shape)
print("pipeline history events:", pipeline_histories.shape)
print(
    "unique master segment_num:",
    pipeline_masters["segment_num"].nunique(),
)
print(
    "duplicate master segment_num:",
    pipeline_masters["segment_num"].duplicated().sum(),
)

history_date_coverage = (
    pipeline_histories["history_date"].notna().mean()
)
unknown_history_codes = (
    pipeline_histories.loc[
        pipeline_histories["history_code"].notna()
        & ~pipeline_histories["history_code_documented"],
        "history_code",
    ]
    .value_counts()
)

print(
    "pipeline history date coverage:",
    round(history_date_coverage, 4),
)
print(
    "undocumented history-code rows:",
    int(unknown_history_codes.sum()),
)

assert history_date_coverage > 0.99, (
    "Pipeline Histories dates were not parsed correctly. "
    f"Coverage={history_date_coverage:.4f}"
)

assert not pipeline_masters["segment_num"].duplicated().any(), (
    "Pipeline Masters contains duplicate segment_num values. "
    "Inspect source before BOEM geometry join."
)

display(
    pipeline_masters[
        [
            "segment_num",
            "seg_length",
            "status_code",
            "prod_code",
            "is_crude_oil_related",
            "aban_type",
        ]
    ].head()
)

display(
    pipeline_histories[
        [
            column
            for column in [
                "segment_num",
                "history_date",
                "history_code",
                "history_event",
                "history_code_documented",
                "service_state_signal",
            ]
            if column in pipeline_histories.columns
        ]
    ].head(20)
)

if not unknown_history_codes.empty:
    print("Undocumented BSEE history codes preserved as raw codes:")
    display(unknown_history_codes)

display(
    pipeline_masters["status_code"]
    .value_counts(dropna=False)
    .head(15)
)

display(
    pipeline_masters["prod_code"]
    .value_counts(dropna=False)
    .head(20)
)


pipeline masters: (21314, 46)
pipeline history events: (85139, 8)
unique master segment_num: 21314
duplicate master segment_num: 0
pipeline history date coverage: 1.0
undocumented history-code rows: 779


,segment_num,seg_length,status_code,prod_code,is_crude_oil_related,aban_type
0,1,9400,ABN,BLKG,False,INPLACE
1,2,12500,ABN,BLKO,True,INPLACE
2,3,10560,ABN,G/C,False,<NA>
3,4,27148,ABN,OIL,True,<NA>
4,6,65848,ACT,OIL,True,<NA>


,segment_num,history_date,history_code,history_event,history_code_documented,service_state_signal
0,1,2005-05-11,B,abandoned_or_proposed_abandonment,True,abandonment_related
1,1,2025-09-17,<NA>,NaN,False,<NA>
2,2,1989-02-01,B,abandoned_or_proposed_abandonment,True,abandonment_related
3,4,2001-04-16,E,miscellaneous,True,<NA>
4,4,2025-09-17,<NA>,NaN,False,<NA>
5,6,2026-10-16,V,NaN,False,<NA>
6,6,2024-08-22,E,miscellaneous,True,<NA>
7,6,2025-08-29,E,miscellaneous,True,<NA>
8,6,2025-04-23,E,miscellaneous,True,<NA>
9,6,2021-05-11,S,service_change,True,<NA>


Undocumented BSEE history codes preserved as raw codes:


history_code
G    375
N    205
V     82
J     26
W     24
K     24
Q     19
Z     16
X      7
0      1
Name: count, dtype: int64[pyarrow]

status_code
ABN     10251
ACT      3932
CNCL     1966
OUT      1479
REM      1332
COMB     1116
PABN      523
PROP      361
A/C       187
PREM      143
<NA>       23
DRFT        1
Name: count, dtype: int64[pyarrow]

prod_code
BLKO    5295
BLKG    5230
GAS     2595
LIFT    2148
OIL     1287
G/C     1274
H2O      386
SPLY     381
UMB      330
FLG      295
G/O      290
UBEH     252
SERV     212
UMBH     197
TEST     148
CBLC     135
CSNG     116
BLOH      92
METH      87
COND      80
Name: count, dtype: int64[pyarrow]

In [13]:
import json

geometry_rows = []

for feature in pipelines_geojson["features"]:
    properties = feature.get("properties") or {}

    geometry_rows.append(
        {
            "segment_num": pd.to_numeric(
                properties.get("SEGMENT_NUM"),
                errors="coerce",
            ),
            "boem_objectid": properties.get("OBJECTID"),
            "geometry_json": json.dumps(
                feature.get("geometry"),
                ensure_ascii=False,
            ),
        }
    )

pipeline_geometry = pd.DataFrame(geometry_rows)

pipeline_geometry["segment_num"] = pd.to_numeric(
    pipeline_geometry["segment_num"],
    errors="coerce",
).astype("Int64")

pipeline_geometry = (
    pipeline_geometry
    .dropna(subset=["segment_num"])
    .drop_duplicates(
        subset=["segment_num"],
        keep="last",
    )
    .reset_index(drop=True)
)

print(
    "BOEM geometry segments:",
    len(pipeline_geometry),
)
print(
    "BSEE master segments:",
    len(pipeline_masters),
)

master_ids = set(
    pipeline_masters["segment_num"].astype(int)
)
geometry_ids = set(
    pipeline_geometry["segment_num"].astype(int)
)

matched_segment_ids = master_ids & geometry_ids

geometry_to_master_coverage = (
    len(matched_segment_ids) / len(geometry_ids)
    if geometry_ids
    else float("nan")
)

print(
    "matched segment IDs:",
    len(matched_segment_ids),
)
master_to_geometry_coverage = (
    len(matched_segment_ids) / len(master_ids)
    if master_ids
    else float("nan")
)

print(
    "master-to-geometry ID coverage:",
    round(master_to_geometry_coverage, 4),
)
print(
    "BOEM-geometry-to-master coverage:",
    round(geometry_to_master_coverage, 4),
)

offshore_pipelines = pipeline_masters.merge(
    pipeline_geometry,
    on="segment_num",
    how="left",
    validate="one_to_one",
)

crude_offshore_pipelines = offshore_pipelines[
    offshore_pipelines[
        "is_crude_oil_related"
    ].fillna(False)
].copy()

geometry_coverage = (
    offshore_pipelines["geometry_json"]
    .notna()
    .mean()
)
active_geometry_coverage = (
    offshore_pipelines.loc[
        offshore_pipelines["status_code"].eq("ACT"),
        "geometry_json",
    ]
    .notna()
    .mean()
)
active_crude_geometry_coverage = (
    crude_offshore_pipelines.loc[
        crude_offshore_pipelines["status_code"].eq("ACT"),
        "geometry_json",
    ]
    .notna()
    .mean()
)

print(
    "master pipelines:",
    len(offshore_pipelines),
)
print(
    "crude/oil-related pipelines:",
    len(crude_offshore_pipelines),
)
print(
    "geometry coverage:",
    round(geometry_coverage, 4),
)
print(
    "active pipeline geometry coverage:",
    round(active_geometry_coverage, 4),
)
print(
    "active crude pipeline geometry coverage:",
    round(active_crude_geometry_coverage, 4),
)

display(
    offshore_pipelines[
        [
            column
            for column in [
                "segment_num",
                "status_code",
                "prod_code",
                "is_crude_oil_related",
                "geometry_json",
            ]
            if column in offshore_pipelines.columns
        ]
    ].head()
)


BOEM geometry segments: 17411
BSEE master segments: 21314
matched segment IDs: 17357
master-to-geometry ID coverage: 0.8143
BOEM-geometry-to-master coverage: 0.9969
master pipelines: 21314
crude/oil-related pipelines: 7030
geometry coverage: 0.8143
active pipeline geometry coverage: 0.9708
active crude pipeline geometry coverage: 0.9767


,segment_num,status_code,prod_code,is_crude_oil_related,geometry_json
0,1,ABN,BLKG,False,"{""type"": ""LineString"", ""coordinates"": [[-91.52..."
1,2,ABN,BLKO,True,NaN
2,3,ABN,G/C,False,NaN
3,4,ABN,OIL,True,NaN
4,6,ACT,OIL,True,"{""type"": ""LineString"", ""coordinates"": [[-91.10..."


In [14]:
offshore_pipelines.to_parquet(
    PROCESSED_DIR / "bsee_offshore_pipelines.parquet",
    index=False,
)
crude_offshore_pipelines.to_parquet(
    PROCESSED_DIR / "bsee_offshore_crude_pipelines.parquet",
    index=False,
)
pipeline_histories.to_parquet(
    PROCESSED_DIR / "bsee_pipeline_history_events.parquet",
    index=False,
)


# 5. NOAA / NHC HURDAT2

Atlantic HURDAT2 даёт 6-hour storm track points. Для graph layer нужны:

- storm location;
- wind intensity/category;
- wind radii;
- time.

Сохраняем pinned исходный HURDAT2 `.txt` в `raw`, а parsed 2015–2025 table —
в `processed`.

2026 не считаем «нулевым hurricane year»: final HURDAT release сейчас заканчивается
2025, поэтому 2026 должен быть отмечен как unavailable до отдельного preliminary source.


In [15]:
from src.data.noaa_hurdat import NoaaHurdatClient

noaa = NoaaHurdatClient()

hurdat_text = noaa.download_text()
hurdat_raw_path = (
    RAW_DIR / "nhc_hurdat2_atlantic_1851_2025.txt"
)
hurdat_raw_path.write_text(
    hurdat_text,
    encoding="utf-8",
)

storms_all = noaa.parse(hurdat_text)
storms = storms_all[
    storms_all["year"].between(2015, 2025)
].reset_index(drop=True)

storms["data_status"] = "final_best_track"
storms["point_in_time_safe"] = False

print("raw HURDAT:", hurdat_raw_path)
print(storms.shape)
display(
    storms.groupby("year")["storm_id"].nunique()
)


raw HURDAT: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/raw/infrastructure/nhc_hurdat2_atlantic_1851_2025.txt
(6223, 25)


year
2015    12
2016    16
2017    18
2018    16
2019    20
2020    31
2021    21
2022    16
2023    21
2024    18
2025    13
Name: storm_id, dtype: int64

In [16]:
storms.to_parquet(
    PROCESSED_DIR
    / "nhc_hurdat2_atlantic_2015_2025.parquet",
    index=False,
)


# 6. OpenStreetMap / Overpass — gap-fill candidates only

OSM **не authoritative master**.

Исправления относительно первого варианта:

- refinery query оставляет только `industrial=refinery`;
- убран слишком широкий `industrial=oil`;
- storage теперь называется **tank candidates**, а не terminals;
- abandoned/disused/demolished objects отбрасываются;
- OSM outputs идут в `processed`, потому что это уже parsed data.

Эти объекты нельзя напрямую считать graph facilities: refineries позже матчим к EIA,
а tanks нужно кластеризовать/матчить к terminal master.


In [17]:
from src.data.overpass_client import OverpassClient

overpass = OverpassClient()

OSM_REGIONS = [
    "gulf_coast",
    "midcontinent",
    "west_coast",
    "east_coast",
]


In [18]:
refinery_frames = []

for region in OSM_REGIONS:
    frame = overpass.fetch_refineries(region)
    refinery_frames.append(frame)
    print(region, len(frame))

osm_refinery_candidates = (
    pd.concat(
        refinery_frames,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["osm_type", "osm_id"],
    )
    .reset_index(drop=True)
)

osm_refinery_candidates.to_parquet(
    PROCESSED_DIR / "osm_refinery_candidates.parquet",
    index=False,
)

print("unique refinery candidates:", len(osm_refinery_candidates))
osm_refinery_candidates.head()


gulf_coast 24


Overpass https://overpass-api.de/api/interpreter returned HTTP 502 (attempt 1/10)


midcontinent 18
west_coast 29
east_coast 14
unique refinery candidates: 85


,osm_type,osm_id,name,operator,latitude,longitude,category,region,tags
0,node,585369735,Lyondell-Citgo Houston Refinery,NaN,29.713241,-95.235027,refinery_candidate,gulf_coast,"{""industrial"": ""refinery"", ""landuse"": ""industr..."
1,way,25486964,Pascagoula Refinery,Chevron,30.344800,-88.490333,refinery_candidate,gulf_coast,"{""industrial"": ""refinery"", ""landuse"": ""industr..."
2,way,137844670,Three Rivers Refinery,Valero,28.456253,-98.190714,refinery_candidate,gulf_coast,"{""addr:housenumber"": ""301"", ""addr:street"": ""Le..."
3,way,293095776,The San Antonio Refinery,"The San Antonio Refinery, LLC",29.347641,-98.460018,refinery_candidate,gulf_coast,"{""addr:city"": ""San Antonio"", ""addr:housenumber..."
4,way,528529366,Sweeny Refinery,Phillips 66,29.073201,-95.745980,refinery_candidate,gulf_coast,"{""industrial"": ""refinery"", ""landuse"": ""industr..."


In [19]:
storage_frames = []

for region in OSM_REGIONS:
    frame = overpass.fetch_oil_storage_tanks(region)
    storage_frames.append(frame)
    print(region, len(frame))

osm_oil_storage_tank_candidates = (
    pd.concat(
        storage_frames,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["osm_type", "osm_id"],
    )
    .reset_index(drop=True)
)

osm_oil_storage_tank_candidates.to_parquet(
    PROCESSED_DIR / "osm_oil_storage_tank_candidates.parquet",
    index=False,
)

print(
    "unique explicit oil/crude tank candidates:",
    len(osm_oil_storage_tank_candidates),
)
osm_oil_storage_tank_candidates.head()


gulf_coast 1019
midcontinent 16747
west_coast 150


Overpass https://overpass-api.de/api/interpreter returned HTTP 502 (attempt 1/10)


east_coast 581
unique explicit oil/crude tank candidates: 18470


,osm_type,osm_id,name,operator,latitude,longitude,category,region,tags
0,node,9357078250,None,NaN,27.774739,-97.473531,oil_storage_tank_candidate,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."
1,node,9357078251,None,NaN,27.774737,-97.473484,oil_storage_tank_candidate,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."
2,node,9357078252,None,NaN,27.774743,-97.473435,oil_storage_tank_candidate,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."
3,node,9357078298,None,NaN,27.768970,-97.482879,oil_storage_tank_candidate,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."
4,node,9357078299,None,NaN,27.768966,-97.482914,oil_storage_tank_candidate,gulf_coast,"{""content"": ""oil"", ""man_made"": ""storage_tank"",..."


# 7. Global Energy Monitor GOIT

GEM остаётся основным onshore transmission-pipeline source. Важно отделить:

- current snapshot/ownership для entity linking;
- approximate historical active interval из `start_year*` / stop/cancel/shelved years.

Current 2026 ownership **не используем как historical ownership truth** для 2015–2025.


In [20]:
from src.data.gem_goit import GemGoitParser

GEM_DIR = ROOT / "data" / "external" / "gem"
GEM_DIR.mkdir(parents=True, exist_ok=True)

gem_parser = GemGoitParser(GEM_DIR)
download_zip = GEM_DIR / "gem-download.zip"

if download_zip.exists():
    gem_files = gem_parser.prepare_download_zip(download_zip)
else:
    gem_files = gem_parser.discover_files()

gem_tables = gem_parser.build_tables(gem_files)

for name, frame in gem_tables.items():
    print(name, frame.shape)


pipelines_all (1926, 69)
pipelines_us (421, 69)
pipeline_ownership_all (23996, 13)
pipeline_ownership_us (7595, 13)
entities (28741, 29)
entity_ownership (27792, 7)


In [21]:
gem_pipelines_us = gem_tables["pipelines_us"].copy()

start_columns = [
    column
    for column in (
        "start_year1",
        "start_year2",
        "start_year3",
    )
    if column in gem_pipelines_us.columns
]
end_columns = [
    column
    for column in (
        "stop_year",
        "cancelled_year",
        "shelved_year",
    )
    if column in gem_pipelines_us.columns
]

for column in [*start_columns, *end_columns]:
    gem_pipelines_us[column] = pd.to_numeric(
        gem_pipelines_us[column],
        errors="coerce",
    )

if start_columns:
    gem_pipelines_us["active_from_year"] = (
        gem_pipelines_us[start_columns].min(axis=1)
    )
else:
    gem_pipelines_us["active_from_year"] = pd.NA

if end_columns:
    gem_pipelines_us["active_to_year"] = (
        gem_pipelines_us[end_columns].min(axis=1)
    )
else:
    gem_pipelines_us["active_to_year"] = pd.NA

gem_pipelines_us["is_oil"] = (
    gem_pipelines_us["fuel"]
    .astype("string")
    .str.casefold()
    .eq("oil")
)
gem_pipelines_us["is_currently_operating"] = (
    gem_pipelines_us["status"]
    .astype("string")
    .str.casefold()
    .eq("operating")
)
gem_pipelines_us["source_snapshot"] = "GOIT 2026-06"
gem_pipelines_us["historical_interval_quality"] = (
    "year_level_approximation"
)

gem_oil_pipelines_temporal = gem_pipelines_us[
    gem_pipelines_us["is_oil"]
].copy()

print("U.S. GEM pipelines:", len(gem_pipelines_us))
print("U.S. Oil pipelines:", len(gem_oil_pipelines_temporal))
print(
    "with start year:",
    gem_oil_pipelines_temporal["active_from_year"].notna().sum(),
)


U.S. GEM pipelines: 421
U.S. Oil pipelines: 315
with start year: 255


In [22]:
gem_parser.save_tables(
    gem_tables,
    PROCESSED_DIR,
)

gem_oil_pipelines_temporal.to_parquet(
    PROCESSED_DIR / "gem_oil_pipelines_temporal.parquet",
    index=False,
)

gem_pipeline_ownership_snapshot = (
    gem_tables["pipeline_ownership_us"].copy()
)
gem_pipeline_ownership_snapshot["ownership_snapshot"] = (
    "2026-10"
)
gem_pipeline_ownership_snapshot[
    "point_in_time_historical_safe"
] = False

gem_pipeline_ownership_snapshot.to_parquet(
    PROCESSED_DIR
    / "gem_pipeline_ownership_us_snapshot_2026_10.parquet",
    index=False,
)


# 8. EIA annual refinery capacity — model-ready

EIA individual-refinery workbooks дают annual refinery capacity.

Исправления:

- убираем дублирующий download/parser code из notebook;
- пытаемся дополнить 2015, если workbook доступен;
- PADD 1–5 отделяем от territories/other rows;
- reported missing operating capacity сохраняем и отдельно имитируем как zero;
- считаем idle capacity;
- добавляем `effective_date`;
- добавляем **консервативный** `safe_available_from = Jan 1 следующего года`.

Последнее специально защищает backtest от знания annual snapshot до его публикации.


In [23]:
from src.data.refinery_capacity import (
    RefineryCapacityDownloader,
    RefineryCapacityParser,
)

REFCAP_DIR = RAW_DIR / "eia_refinery_capacity"
REFCAP_DIR.mkdir(parents=True, exist_ok=True)

refinery_downloader = RefineryCapacityDownloader()
refinery_parser = RefineryCapacityParser()

existing_files = refinery_parser.discover_files(
    REFCAP_DIR
)
existing_years = {
    refinery_parser._report_year(path)
    for path in existing_files
}

missing_years = [
    year
    for year in range(2015, 2027)
    if year not in existing_years
]

print("existing years:", sorted(existing_years))
print("missing years before download:", missing_years)

for year in missing_years:
    try:
        path = refinery_downloader.download_year(
            year,
            REFCAP_DIR,
        )
        print("downloaded", path.name)
    except Exception as exc:
        print(
            f"WARNING: could not download EIA refinery {year}: {exc}"
        )

refinery_files = refinery_parser.discover_files(
    REFCAP_DIR
)

print("final files:", [path.name for path in refinery_files])


existing years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
missing years before download: []
final files: ['refcap2015.xls', 'refcap2016.xls', 'refcap2017.xls', 'refcap2018.xls', 'refcap2019.xls', 'refcap2020.xls', 'refcap2021.xls', 'refcap2022.xls', 'refcap2023.xls', 'refcap2024.xlsx', 'refcap2025.xlsx', 'refcap2026.xlsx']


In [24]:
refinery_capacity_long = refinery_parser.build_long(
    refinery_files
)
refinery_year = refinery_parser.build_refinery_year(
    refinery_capacity_long
)
refineries_padd_1_5, refineries_other = (
    refinery_parser.split_main_padds(
        refinery_year
    )
)

duplicate_location_rows = (
    refineries_padd_1_5.duplicated(
        ["report_year", "refinery_location_key"],
        keep=False,
    )
)
duplicate_record_rows = (
    refineries_padd_1_5.duplicated(
        ["report_year", "refinery_key"],
        keep=False,
    )
)

print("long rows:", len(refinery_capacity_long))
print("refinery-year rows:", len(refinery_year))
print("PADD 1-5 rows:", len(refineries_padd_1_5))
print("territory/other rows:", len(refineries_other))
print(
    "years:",
    refinery_year["report_year"].min(),
    "->",
    refinery_year["report_year"].max(),
)
print(
    "rows sharing city/location key:",
    int(duplicate_location_rows.sum()),
)
print(
    "rows sharing full refinery record key:",
    int(duplicate_record_rows.sum()),
)

assert not duplicate_record_rows.any(), (
    "Full EIA refinery source key is still not unique "
    "inside report year."
)

if duplicate_location_rows.any():
    print(
        "\nExpected: some cities contain multiple distinct refineries."
    )
    display(
        refineries_padd_1_5.loc[
            duplicate_location_rows,
            [
                "report_year",
                "refinery_location_key",
                "company_name",
                "corporation",
                "site",
                "state_name",
                "padd",
                "total_operable_capacity_bpcd",
            ],
        ]
        .sort_values(
            [
                "report_year",
                "refinery_location_key",
                "company_name",
            ]
        )
        .head(60)
    )


long rows: 43099
refinery-year rows: 1564
PADD 1-5 rows: 1561
territory/other rows: 3
years: 2015 -> 2026
rows sharing city/location key: 422
rows sharing full refinery record key: 0

Expected: some cities contain multiple distinct refineries.


,report_year,refinery_location_key,company_name,corporation,site,state_name,padd,total_operable_capacity_bpcd
10,2015,alaska|prudhoe bay,BP EXPLORATION ALASKA INC,BP PLC,PRUDHOE BAY,Alaska,5,10500.0
29,2015,alaska|prudhoe bay,CONOCOPHILLIPS ALASKA INC,CONOCOPHILLIPS,PRUDHOE BAY,Alaska,5,15000.0
61,2015,california|bakersfield,KERN OIL & REFINING CO,KERN OIL & REFINING CO,BAKERSFIELD,California,5,26000.0
100,2015,california|bakersfield,SAN JOAQUIN REFINING CO INC,SAN JOAQUIN REFINING CO INC,BAKERSFIELD,California,5,15000.0
98,2015,california|martinez,Shell Oil Products US,ROYAL DUTCH/SHELL GROUP,MARTINEZ,California,5,156400.0
109,2015,california|martinez,TESORO REFINING & MARKETING CO,TESORO CORP,MARTINEZ,California,5,166000.0
93,2015,california|wilmington,PHILLIPS 66 COMPANY,PHILLIPS 66 COMPANY,WILMINGTON,California,5,139000.0
110,2015,california|wilmington,TESORO REFINING & MARKETING CO,TESORO CORP,WILMINGTON,California,5,104500.0
115,2015,louisiana|lake charles,CALCASIEU REFINING CO,TRANSWORLD OIL USA INC,LAKE CHARLES,Louisiana,3,80000.0
82,2015,louisiana|lake charles,CITGO PETROLEUM CORP,PDV AMERICA INC,LAKE CHARLES,Louisiana,3,427800.0


In [25]:
refinery_qa = (
    refineries_padd_1_5.groupby("report_year")
    .agg(
        refinery_rows=("refinery_key", "size"),
        unique_refinery_records=("refinery_key", "nunique"),
        unique_location_keys=("refinery_location_key", "nunique"),
        total_operable_bpcd=(
            "total_operable_capacity_bpcd",
            "sum",
        ),
        operating_bpcd=(
            "operating_capacity_bpcd",
            "sum",
        ),
        imputed_zero_rows=(
            "operating_capacity_imputed_zero",
            "sum",
        ),
        idle_rows=("is_idle", "sum"),
    )
)

refinery_qa


,refinery_rows,unique_refinery_records,unique_location_keys,total_operable_bpcd,operating_bpcd,imputed_zero_rows,idle_rows
report_year,,,,,,,
2015,135,135,115,17967088.0,17767588.0,3,3
2016,137,137,115,18317036.0,18165136.0,2,2
2017,137,137,114,18617027.0,18314127.0,4,4
2018,132,132,111,18598497.0,18566897.0,0,0
2019,132,132,112,18802435.0,18692335.0,3,3
2020,132,132,113,18976085.0,18548985.0,4,4
2021,126,126,108,18127700.0,17719600.0,5,5
2022,127,127,109,17943810.0,17789010.0,5,5
2023,126,126,108,18060369.0,17684549.0,5,5


In [26]:
refinery_capacity_long.to_parquet(
    PROCESSED_DIR / "eia_refinery_capacity_long.parquet",
    index=False,
)
refineries_padd_1_5.to_parquet(
    PROCESSED_DIR / "eia_refineries_model_ready.parquet",
    index=False,
)
refineries_other.to_parquet(
    PROCESSED_DIR / "eia_refineries_territories_other.parquet",
    index=False,
)


# 9. Final QA — что теперь считается готовым

После этой версии ветка создаёт **source layer**, а не финальный weekly feature matrix.

### Historical offshore nodes
- `bsee_platform_history.parquet` — structure lifetime + coordinates;
- `bsee_production_by_structure_2015plus.parquet`;
- `bsee_production_by_complex_month_2015plus.parquet`.

### Historical offshore edges/events
- `bsee_offshore_pipelines.parquet` — geometry + product/current status;
- `bsee_offshore_crude_pipelines.parquet`;
- `bsee_pipeline_history_events.parquet` — dated service timeline.

### Weather shocks
- `nhc_hurdat2_atlantic_2015_2025.parquet`.

### Onshore pipeline graph
- GEM normalized tables;
- `gem_oil_pipelines_temporal.parquet`;
- ownership snapshot explicitly marked as non-point-in-time historical data.

### Refineries
- `eia_refineries_model_ready.parquet` — PADD 1–5, capacity/idle/availability fields;
- territories stored separately.

### OSM
- only gap-fill candidates, **not graph truth**.

## Следующий этап

На следующей ветке из этих source tables строим weekly features / graph snapshots:

`storm -> affected active platforms -> exposed lagged BOPD -> PADD / crude balance`

и связываем news entities с refinery / pipeline / company nodes.


In [27]:
print("Infrastructure source-layer build completed.")
print("Processed directory:", PROCESSED_DIR)

expected_outputs = [
    "bsee_platform_history.parquet",
    "bsee_production_by_structure_2015plus.parquet",
    "bsee_production_by_complex_month_2015plus.parquet",
    "bsee_offshore_pipelines.parquet",
    "bsee_offshore_crude_pipelines.parquet",
    "bsee_pipeline_history_events.parquet",
    "nhc_hurdat2_atlantic_2015_2025.parquet",
    "osm_refinery_candidates.parquet",
    "osm_oil_storage_tank_candidates.parquet",
    "gem_oil_pipelines_temporal.parquet",
    "eia_refineries_model_ready.parquet",
]

missing_outputs = []

for filename in expected_outputs:
    path = PROCESSED_DIR / filename
    exists = path.exists()
    print(f"{filename}: {'OK' if exists else 'MISSING'}")
    if not exists:
        missing_outputs.append(filename)

assert not missing_outputs, (
    f"Missing processed outputs: {missing_outputs}"
)

# Core source-layer quality gates.
assert platform_history["has_coordinates"].mean() > 0.98
assert historical_platform_coverage > 0.95
assert pipeline_histories["history_date"].notna().mean() > 0.99
assert geometry_to_master_coverage > 0.99
assert active_geometry_coverage > 0.95
assert active_crude_geometry_coverage > 0.95
assert refinery_year["report_year"].min() == 2015
assert refinery_year["report_year"].max() == 2026
assert set(
    refineries_padd_1_5["padd"].dropna().astype(int).unique()
).issubset({1, 2, 3, 4, 5})
assert not refineries_padd_1_5.duplicated(
    ["report_year", "refinery_key"]
).any()

# refinery_location_key is intentionally NOT unique:
# multiple physical refineries can be located in the same city/site label.
assert refineries_padd_1_5[
    "total_operable_capacity_bpcd"
].notna().all()

valid_capacity_share = (
    refineries_padd_1_5["operating_capacity_share"].isna()
    | refineries_padd_1_5["operating_capacity_share"].between(
        0,
        1.000001,
    )
)
assert valid_capacity_share.all()

# These are files from older notebook versions and should not be used.
legacy_raw_files = [
    RAW_DIR / "osm_refineries.parquet",
    RAW_DIR / "osm_oil_storage_terminals.parquet",
    RAW_DIR / "nhc_hurdat2_atlantic_2015_2025.parquet",
]

legacy_processed_files = [
    PROCESSED_DIR / "boem_platforms.parquet",
    PROCESSED_DIR / "bsee_production_by_platform.parquet",
    PROCESSED_DIR / "eia_refinery_capacity_long_2016_2026.parquet",
    PROCESSED_DIR / "eia_refineries_2016_2026.parquet",
]

legacy_existing = [
    path
    for path in [
        *legacy_raw_files,
        *legacy_processed_files,
    ]
    if path.exists()
]

if legacy_existing:
    print("\nLEGACY FILES FOUND — safe to delete:")
    for path in legacy_existing:
        print(path.relative_to(ROOT))
else:
    print("\nNo known legacy data files found.")


Infrastructure source-layer build completed.
Processed directory: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/infrastructure
bsee_platform_history.parquet: OK
bsee_production_by_structure_2015plus.parquet: OK
bsee_production_by_complex_month_2015plus.parquet: OK
bsee_offshore_pipelines.parquet: OK
bsee_offshore_crude_pipelines.parquet: OK
bsee_pipeline_history_events.parquet: OK
nhc_hurdat2_atlantic_2015_2025.parquet: OK
osm_refinery_candidates.parquet: OK
osm_oil_storage_tank_candidates.parquet: OK
gem_oil_pipelines_temporal.parquet: OK
eia_refineries_model_ready.parquet: OK

No known legacy data files found.
